<a href="https://colab.research.google.com/github/maneesha-bogahawatta/J26-DS-320/blob/member4%2Fcritical-incidents/components/c4_critical_incidents/fall_branch/C4_fall_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive/J26-DS-320'):
    print(root)

/content/drive/MyDrive/J26-DS-320
/content/drive/MyDrive/J26-DS-320/datasets
/content/drive/MyDrive/J26-DS-320/datasets/urfd
/content/drive/MyDrive/J26-DS-320/datasets/urfd/Falls
/content/drive/MyDrive/J26-DS-320/datasets/urfd/Falls/fall-30-cam0-rgb
/content/drive/MyDrive/J26-DS-320/datasets/urfd/Falls/fall-25-cam0-rgb
/content/drive/MyDrive/J26-DS-320/datasets/urfd/Falls/fall-20-cam0-rgb
/content/drive/MyDrive/J26-DS-320/datasets/urfd/Falls/fall-27-cam0-rgb
/content/drive/MyDrive/J26-DS-320/datasets/urfd/Falls/fall-16-cam0-rgb
/content/drive/MyDrive/J26-DS-320/datasets/urfd/Falls/fall-15-cam0-rgb
/content/drive/MyDrive/J26-DS-320/datasets/urfd/Falls/fall-24-cam0-rgb
/content/drive/MyDrive/J26-DS-320/datasets/urfd/Falls/fall-26-cam0-rgb
/content/drive/MyDrive/J26-DS-320/datasets/urfd/Falls/fall-28-cam0-rgb
/content/drive/MyDrive/J26-DS-320/datasets/urfd/Falls/fall-23-cam0-rgb
/content/drive/MyDrive/J26-DS-320/datasets/urfd/Falls/fall-17-cam0-rgb
/content/drive/MyDrive/J26-DS-320/datase

In [4]:
!pip install mediapipe opencv-python-headless -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 11.0 MB/s eta 0:00:00


In [5]:
import os

dataset_path = '/content/drive/MyDrive/J26-DS-320/datasets/urfd'
falls_path = os.path.join(dataset_path, 'Falls')
normal_path = os.path.join(dataset_path, 'Normal')

# Filter out Mac's hidden "._" ghost files
def clean_list(folder_path):
    return sorted([f for f in os.listdir(folder_path) if not f.startswith('.')])

fall_folders = clean_list(falls_path)
normal_folders = clean_list(normal_path)

print("Number of fall sequences:", len(fall_folders))
print("Number of normal sequences:", len(normal_folders))

# Look inside the first REAL fall folder
first_fall = fall_folders[0]
first_fall_path = os.path.join(falls_path, first_fall)
print("\nFirst fall folder name:", first_fall)

# Also filter ghost files inside the folder
image_files = clean_list(first_fall_path)
print("Number of images inside:", len(image_files))
print("First 5 image names:", image_files[:5])

Number of fall sequences: 30
Number of normal sequences: 40

First fall folder name: fall-01-cam0-rgb
Number of images inside: 160
First 5 image names: ['fall-01-cam0-rgb-001.png', 'fall-01-cam0-rgb-002.png', 'fall-01-cam0-rgb-003.png', 'fall-01-cam0-rgb-004.png', 'fall-01-cam0-rgb-005.png']


In [6]:
!pip install mediapipe -q

In [7]:
import mediapipe as mp
print("Version:", mp.__version__)
print("Has solutions:", hasattr(mp, 'solutions'))

Version: 1.1.0
Has solutions: False


In [8]:
!wget -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task
print("Model downloaded")

Model downloaded


In [9]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Set up the new pose detector
base_options = python.BaseOptions(model_asset_path='pose_landmarker_lite.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=False)
detector = vision.PoseLandmarker.create_from_options(options)


def extract_pose_features(image_path):
    """
    Read one image, run pose estimation, return key features.
    Returns None if no person detected.
    """
    frame = cv2.imread(image_path)
    if frame is None:
        return None

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    result = detector.detect(mp_image)

    if not result.pose_landmarks or len(result.pose_landmarks) == 0:
        return None

    landmarks = result.pose_landmarks[0]  # first detected person

    # Key body points (same indices as before)
    nose_y = landmarks[0].y
    left_shoulder_y = landmarks[11].y
    right_shoulder_y = landmarks[12].y
    left_hip_y = landmarks[23].y
    right_hip_y = landmarks[24].y

    shoulder_y = (left_shoulder_y + right_shoulder_y) / 2
    hip_y = (left_hip_y + right_hip_y) / 2

    # Bounding box from all landmarks
    xs = [lm.x for lm in landmarks]
    ys = [lm.y for lm in landmarks]

    width = max(xs) - min(xs)
    height = max(ys) - min(ys)
    aspect_ratio = width / height if height > 0.01 else 0

    return {
        "nose_y": nose_y,
        "shoulder_y": shoulder_y,
        "hip_y": hip_y,
        "aspect_ratio": aspect_ratio,
        "width": width,
        "height": height,
    }


# TEST on the first fall sequence
test_folder = os.path.join(falls_path, fall_folders[0])
test_images = clean_list(test_folder)

print(f"Testing on {len(test_images)} frames from {fall_folders[0]}\n")

for i, img_name in enumerate(test_images[::20]):
    img_path = os.path.join(test_folder, img_name)
    features = extract_pose_features(img_path)
    if features:
        print(f"Frame {img_name}: aspect_ratio={features['aspect_ratio']:.3f}, nose_y={features['nose_y']:.3f}, hip_y={features['hip_y']:.3f}")
    else:
        print(f"Frame {img_name}: NO PERSON DETECTED")

Testing on 160 frames from fall-01-cam0-rgb

Frame fall-01-cam0-rgb-001.png: aspect_ratio=0.273, nose_y=0.257, hip_y=0.461
Frame fall-01-cam0-rgb-021.png: aspect_ratio=0.328, nose_y=0.257, hip_y=0.470
Frame fall-01-cam0-rgb-041.png: aspect_ratio=0.208, nose_y=0.251, hip_y=0.457
Frame fall-01-cam0-rgb-061.png: aspect_ratio=0.305, nose_y=0.239, hip_y=0.467
Frame fall-01-cam0-rgb-081.png: aspect_ratio=0.293, nose_y=0.235, hip_y=0.473
Frame fall-01-cam0-rgb-101.png: aspect_ratio=0.502, nose_y=0.503, hip_y=0.625
Frame fall-01-cam0-rgb-121.png: aspect_ratio=0.392, nose_y=0.684, hip_y=0.799
Frame fall-01-cam0-rgb-141.png: NO PERSON DETECTED


In [20]:
import numpy as np
from tqdm import tqdm
import pickle

def process_sequence(folder_path, label):
    """
    Process one video sequence folder.
    Returns a list of feature dicts (one per frame), or None if too few valid frames.
    """
    images = clean_list(folder_path)
    sequence_features = []

    for img_name in images:
        img_path = os.path.join(folder_path, img_name)
        features = extract_pose_features(img_path)
        if features:
            sequence_features.append(features)

    if len(sequence_features) < 10:  # too few valid frames, skip
        return None

    return {
        "features": sequence_features,
        "label": label,  # 1 = fall, 0 = normal
        "num_frames": len(sequence_features),
    }


all_sequences = []

# Process fall sequences
print("Processing FALL sequences...")
for folder_name in tqdm(fall_folders):
    folder_path = os.path.join(falls_path, folder_name)
    result = process_sequence(folder_path, label=1)
    if result:
        result["name"] = folder_name
        all_sequences.append(result)

# Process normal sequences
print("\nProcessing NORMAL sequences...")
for folder_name in tqdm(normal_folders):
    folder_path = os.path.join(normal_path, folder_name)
    result = process_sequence(folder_path, label=0)
    if result:
        result["name"] = folder_name
        all_sequences.append(result)

print(f"\nTotal sequences processed: {len(all_sequences)}")
print(f"Fall sequences: {sum(1 for s in all_sequences if s['label'] == 1)}")
print(f"Normal sequences: {sum(1 for s in all_sequences if s['label'] == 0)}")

# Save to Google Drive so we don't have to redo this every time
save_path = '/content/drive/MyDrive/J26-DS-320/datasets/urfd_features.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(all_sequences, f)

print(f"\nSaved to: {save_path}")

Processing FALL sequences...


100%|██████████| 30/30 [03:30<00:00,  7.03s/it]



Processing NORMAL sequences...


100%|██████████| 40/40 [09:20<00:00, 14.01s/it]


Total sequences processed: 70
Fall sequences: 30
Normal sequences: 40

Saved to: /content/drive/MyDrive/J26-DS-320/datasets/urfd_features.pkl


In [11]:
def run_baseline_on_sequence(sequence_features, aspect_threshold=0.5, confirm_frames=5):
    """
    Simple rule-based fall detector.
    Rule: if aspect_ratio > threshold for `confirm_frames` consecutive frames → FALL
    """
    fall_counter = 0
    max_counter = 0

    for f in sequence_features:
        if f["aspect_ratio"] > aspect_threshold:
            fall_counter += 1
        else:
            fall_counter = max(0, fall_counter - 1)
        max_counter = max(max_counter, fall_counter)

    detected_fall = max_counter >= confirm_frames
    confidence = min(max_counter / confirm_frames, 1.0)

    return detected_fall, confidence


# Run baseline on all sequences
results = []

for seq in all_sequences:
    detected, confidence = run_baseline_on_sequence(seq["features"])
    ground_truth = seq["label"]  # 1 = fall, 0 = normal

    results.append({
        "name": seq["name"],
        "ground_truth": "fall" if ground_truth == 1 else "normal",
        "predicted": "fall" if detected else "normal",
        "correct": (detected == 1) == (ground_truth == 1),
        "confidence": round(confidence, 3),
    })

# Print results table
print(f"{'Sequence':30s} {'Truth':8s} {'Predicted':10s} {'Correct':8s} {'Confidence'}")
print("-" * 75)
for r in results:
    status = "✓" if r["correct"] else "✗"
    print(f"{r['name']:30s} {r['ground_truth']:8s} {r['predicted']:10s} {status:8s} {r['confidence']}")

Sequence                       Truth    Predicted  Correct  Confidence
---------------------------------------------------------------------------
fall-01-cam0-rgb               fall     fall       ✓        1.0
fall-02-cam0-rgb               fall     fall       ✓        1.0
fall-03-cam0-rgb               fall     fall       ✓        1.0
fall-04-cam0-rgb               fall     fall       ✓        1.0
fall-05-cam0-rgb               fall     fall       ✓        1.0
fall-06-cam0-rgb               fall     fall       ✓        1.0
fall-07-cam0-rgb               fall     fall       ✓        1.0
fall-08-cam0-rgb               fall     fall       ✓        1.0
fall-09-cam0-rgb               fall     fall       ✓        1.0
fall-10-cam0-rgb               fall     fall       ✓        1.0
fall-11-cam0-rgb               fall     fall       ✓        1.0
fall-12-cam0-rgb               fall     fall       ✓        1.0
fall-13-cam0-rgb               fall     normal     ✗        0.4
fall-14-cam0-rgb     

In [12]:
def calculate_metrics(results):
    tp = sum(1 for r in results if r["ground_truth"] == "fall" and r["predicted"] == "fall")
    fp = sum(1 for r in results if r["ground_truth"] == "normal" and r["predicted"] == "fall")
    fn = sum(1 for r in results if r["ground_truth"] == "fall" and r["predicted"] == "normal")
    tn = sum(1 for r in results if r["ground_truth"] == "normal" and r["predicted"] == "normal")

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    false_alarm_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
    accuracy = (tp + tn) / len(results)

    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "false_alarm_rate": round(false_alarm_rate, 4),
        "accuracy": round(accuracy, 4),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }


metrics = calculate_metrics(results)

print("=" * 50)
print("BASELINE RESULTS (aspect_ratio > 0.5)")
print("=" * 50)
print(f"True Positives:    {metrics['tp']}  (correctly caught falls)")
print(f"False Positives:   {metrics['fp']}  (false alarms)")
print(f"False Negatives:   {metrics['fn']}  (missed falls)")
print(f"True Negatives:    {metrics['tn']}  (correctly ignored normal)")
print("-" * 50)
print(f"Precision:         {metrics['precision']}")
print(f"Recall:            {metrics['recall']}")
print(f"F1 Score:          {metrics['f1']}")
print(f"False Alarm Rate:  {metrics['false_alarm_rate']}")
print(f"Accuracy:          {metrics['accuracy']}")
print("=" * 50)

BASELINE RESULTS (aspect_ratio > 0.5)
True Positives:    27  (correctly caught falls)
False Positives:   23  (false alarms)
False Negatives:   3  (missed falls)
True Negatives:    17  (correctly ignored normal)
--------------------------------------------------
Precision:         0.54
Recall:            0.9
F1 Score:          0.675
False Alarm Rate:  0.575
Accuracy:          0.6286


In [13]:
import json

baseline_summary = {
    "method": "Aspect ratio threshold (0.5) + 5-frame confirmation",
    "metrics": metrics,
    "per_sequence_results": results,
}

save_path = '/content/drive/MyDrive/J26-DS-320/datasets/baseline_results.json'
with open(save_path, 'w') as f:
    json.dump(baseline_summary, f, indent=2)

print(f"Baseline results saved to: {save_path}")
print(f"\nFinal baseline F1 score: {metrics['f1']}")

Baseline results saved to: /content/drive/MyDrive/J26-DS-320/datasets/baseline_results.json

Final baseline F1 score: 0.675


In [14]:
import numpy as np

# --- Turn each sequence into a fixed-length array ---

MAX_FRAMES = 100   # pad/cut every sequence to this many frames
FEATURE_NAMES = ["nose_y", "shoulder_y", "hip_y", "aspect_ratio", "width", "height"]

def sequence_to_array(seq_features, max_frames=MAX_FRAMES):
    """
    Convert a list of feature dicts into a (max_frames, num_features) array.
    Pads with zeros if shorter, truncates if longer.
    """
    arr = np.zeros((max_frames, len(FEATURE_NAMES)), dtype=np.float32)
    actual_len = min(len(seq_features), max_frames)

    for i in range(actual_len):
        f = seq_features[i]
        arr[i] = [f[name] for name in FEATURE_NAMES]

    return arr, actual_len


# Build X (features) and y (labels) for every sequence
X = []
y = []
names = []
lengths = []

for seq in all_sequences:
    arr, actual_len = sequence_to_array(seq["features"])
    X.append(arr)
    y.append(seq["label"])
    names.append(seq["name"])
    lengths.append(actual_len)

X = np.array(X)   # shape: (70, 100, 6)
y = np.array(y)   # shape: (70,)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Falls:", sum(y == 1), "Normal:", sum(y == 0))

X shape: (70, 100, 6)
y shape: (70,)
Falls: 30 Normal: 40


In [15]:
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temp (which becomes val+test)
X_train, X_temp, y_train, y_temp, names_train, names_temp = train_test_split(
    X, y, names, test_size=0.3, stratify=y, random_state=42
)

# Second split: split temp into validation and test (50/50 of the 30%)
X_val, X_test, y_val, y_test, names_val, names_test = train_test_split(
    X_temp, y_temp, names_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"Train: {len(X_train)} sequences ({sum(y_train==1)} falls, {sum(y_train==0)} normal)")
print(f"Val:   {len(X_val)} sequences ({sum(y_val==1)} falls, {sum(y_val==0)} normal)")
print(f"Test:  {len(X_test)} sequences ({sum(y_test==1)} falls, {sum(y_test==0)} normal)")

Train: 49 sequences (21 falls, 28 normal)
Val:   10 sequences (4 falls, 6 normal)
Test:  11 sequences (5 falls, 6 normal)


In [16]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


class FallCNNLSTM(nn.Module):
    """
    CNN-LSTM for fall detection.
    Input:  (batch, 100 frames, 6 features)
    Output: (batch, 2) -> [not_fall_prob, fall_prob]
    """
    def __init__(self, num_features=6, cnn_channels=[32, 64, 128], lstm_hidden=128, lstm_layers=2):
        super().__init__()

        # CNN layers operate along the time dimension
        self.conv1 = nn.Conv1d(num_features, cnn_channels[0], kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(cnn_channels[0], cnn_channels[1], kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(cnn_channels[1], cnn_channels[2], kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

        # LSTM learns temporal patterns from CNN features
        self.lstm = nn.LSTM(
            input_size=cnn_channels[2],
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=0.3,
        )

        # Final classifier
        self.fc1 = nn.Linear(lstm_hidden, 64)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        # x shape: (batch, 100, 6) -> need (batch, 6, 100) for Conv1d
        x = x.permute(0, 2, 1)

        x = self.relu(self.conv1(x))
        x = self.dropout(x)
        x = self.relu(self.conv2(x))
        x = self.dropout(x)
        x = self.relu(self.conv3(x))
        x = self.dropout(x)

        # back to (batch, 100, channels) for LSTM
        x = x.permute(0, 2, 1)

        lstm_out, (h_n, c_n) = self.lstm(x)
        # Use the last hidden state
        last_hidden = h_n[-1]

        x = self.relu(self.fc1(last_hidden))
        x = self.fc2(x)
        return x


model = FallCNNLSTM().to(device)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

Using device: cpu
FallCNNLSTM(
  (conv1): Conv1d(6, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv2): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv3): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=2, bias=True)
)

Total parameters: 304,098


In [17]:
from torch.utils.data import Dataset, DataLoader

class FallDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = FallDataset(X_train, y_train)
val_dataset = FallDataset(X_val, y_val)
test_dataset = FallDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 7
Val batches: 2
Test batches: 2


In [18]:
import numpy as np
import torch
import torch.nn as nn

print("numpy version:", np.__version__)
print("torch version:", torch.__version__)

numpy version: 2.1.3
torch version: 2.11.0+cpu


In [19]:
import torch.optim as optim
from sklearn.metrics import precision_score, recall_score, f1_score

# Class weights to handle imbalance (28 normal vs 21 fall in training set)
class_counts = np.bincount(y_train)
class_weights = torch.tensor([1.0 / c for c in class_counts], dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)

EPOCHS = 50
best_val_f1 = 0
best_model_state = None
patience = 10
patience_counter = 0

history = {"train_loss": [], "val_loss": [], "val_f1": []}

for epoch in range(EPOCHS):
    # --- Training ---
    model.train()
    train_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # --- Validation ---
    model.eval()
    val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    val_loss /= len(val_loader)
    val_f1 = f1_score(all_labels, all_preds, zero_division=0)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = model.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")

    # Early stopping
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
        break

print(f"\nBest validation F1: {best_val_f1:.4f}")

# Load best model
model.load_state_dict(best_model_state)

Epoch  1/50 | Train Loss: 0.6943 | Val Loss: 0.6843 | Val F1: 0.0000
Epoch  2/50 | Train Loss: 0.6885 | Val Loss: 0.6742 | Val F1: 0.0000
Epoch  3/50 | Train Loss: 0.6800 | Val Loss: 0.6454 | Val F1: 0.0000
Epoch  4/50 | Train Loss: 0.6724 | Val Loss: 0.5544 | Val F1: 0.0000
Epoch  5/50 | Train Loss: 0.6569 | Val Loss: 0.4817 | Val F1: 0.7500
Epoch  6/50 | Train Loss: 0.5956 | Val Loss: 0.4665 | Val F1: 0.7500
Epoch  7/50 | Train Loss: 0.4628 | Val Loss: 0.3843 | Val F1: 0.7500
Epoch  8/50 | Train Loss: 0.3898 | Val Loss: 0.4280 | Val F1: 0.7500
Epoch  9/50 | Train Loss: 0.3632 | Val Loss: 0.4226 | Val F1: 0.7500
Epoch 10/50 | Train Loss: 0.3131 | Val Loss: 0.4298 | Val F1: 0.7500
Epoch 11/50 | Train Loss: 0.3008 | Val Loss: 0.4155 | Val F1: 0.7500
Epoch 12/50 | Train Loss: 0.3049 | Val Loss: 0.4193 | Val F1: 0.7500
Epoch 13/50 | Train Loss: 0.5349 | Val Loss: 0.4297 | Val F1: 0.7500
Epoch 14/50 | Train Loss: 0.3253 | Val Loss: 0.4707 | Val F1: 0.7500
Epoch 15/50 | Train Loss: 0.4761 |

<All keys matched successfully>

In [21]:
import pickle

# Load the features we already extracted and saved
load_path = '/content/drive/MyDrive/J26-DS-320/datasets/urfd_features.pkl'
with open(load_path, 'rb') as f:
    all_sequences = pickle.load(f)

print(f"Loaded {len(all_sequences)} sequences from saved file")
print(f"Falls: {sum(1 for s in all_sequences if s['label'] == 1)}")
print(f"Normal: {sum(1 for s in all_sequences if s['label'] == 0)}")

Loaded 70 sequences from saved file
Falls: 30
Normal: 40


In [22]:
model.eval()
test_preds = []
test_labels = []
test_confidences = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = model(batch_X)
        probs = torch.softmax(outputs, dim=1)

        preds = torch.argmax(outputs, dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(batch_y.cpu().numpy())
        test_confidences.extend(probs[:, 1].cpu().numpy())  # fall probability

test_preds = np.array(test_preds)
test_labels = np.array(test_labels)

# Per-sequence results table
print(f"{'Sequence':30s} {'Truth':8s} {'Predicted':10s} {'Correct':8s} {'Fall Prob'}")
print("-" * 75)
for i, name in enumerate(names_test):
    truth = "fall" if test_labels[i] == 1 else "normal"
    pred = "fall" if test_preds[i] == 1 else "normal"
    correct = "✓" if test_labels[i] == test_preds[i] else "✗"
    print(f"{name:30s} {truth:8s} {pred:10s} {correct:8s} {test_confidences[i]:.3f}")

# Final metrics
test_precision = precision_score(test_labels, test_preds, zero_division=0)
test_recall = recall_score(test_labels, test_preds, zero_division=0)
test_f1 = f1_score(test_labels, test_preds, zero_division=0)

tp = sum((test_labels == 1) & (test_preds == 1))
fp = sum((test_labels == 0) & (test_preds == 1))
fn = sum((test_labels == 1) & (test_preds == 0))
tn = sum((test_labels == 0) & (test_preds == 0))
false_alarm_rate = fp / (fp + tn) if (fp + tn) > 0 else 0

print("\n" + "=" * 50)
print("CNN-LSTM TEST SET RESULTS")
print("=" * 50)
print(f"True Positives:    {tp}")
print(f"False Positives:   {fp}")
print(f"False Negatives:   {fn}")
print(f"True Negatives:    {tn}")
print("-" * 50)
print(f"Precision:         {test_precision:.4f}")
print(f"Recall:            {test_recall:.4f}")
print(f"F1 Score:          {test_f1:.4f}")
print(f"False Alarm Rate:  {false_alarm_rate:.4f}")
print("=" * 50)
print(f"\nBASELINE F1 was:   0.6750")
print(f"CNN-LSTM F1 is:    {test_f1:.4f}")
print(f"Improvement:       {test_f1 - 0.675:+.4f}")

Sequence                       Truth    Predicted  Correct  Fall Prob
---------------------------------------------------------------------------
adl-11-cam0-rgb                normal   normal     ✓        0.233
adl-29-cam0-rgb                normal   normal     ✓        0.232
adl-15-cam0-rgb                normal   normal     ✓        0.232
fall-28-cam0-rgb               fall     fall       ✓        0.969
adl-33-cam0-rgb                normal   normal     ✓        0.242
fall-07-cam0-rgb               fall     normal     ✗        0.232
fall-04-cam0-rgb               fall     fall       ✓        0.969
adl-23-cam0-rgb                normal   normal     ✓        0.336
adl-36-cam0-rgb                normal   normal     ✓        0.230
fall-14-cam0-rgb               fall     fall       ✓        0.969
fall-18-cam0-rgb               fall     fall       ✓        0.969

CNN-LSTM TEST SET RESULTS
True Positives:    4
False Positives:   0
False Negatives:   1
True Negatives:    6
-----------------

In [23]:
import json

# Save the trained model weights
model_save_path = '/content/drive/MyDrive/J26-DS-320/datasets/fall_cnn_lstm_v1.pt'
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to: {model_save_path}")

# Save the full results summary
results_summary = {
    "model": "CNN-LSTM (3 conv layers + 2 LSTM layers)",
    "training": {
        "epochs_trained": len(history["train_loss"]),
        "best_val_f1": float(best_val_f1),
        "train_sequences": len(X_train),
        "val_sequences": len(X_val),
        "test_sequences": len(X_test),
    },
    "test_metrics": {
        "precision": float(test_precision),
        "recall": float(test_recall),
        "f1": float(test_f1),
        "false_alarm_rate": float(false_alarm_rate),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    },
    "baseline_f1": 0.675,
    "improvement": float(test_f1 - 0.675),
    "missed_sequences": [names_test[i] for i in range(len(test_labels)) if test_labels[i] != test_preds[i] and test_labels[i] == 1],
}

results_path = '/content/drive/MyDrive/J26-DS-320/datasets/cnn_lstm_results.json'
with open(results_path, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"Results saved to: {results_path}")
print(f"\nSummary: {json.dumps(results_summary, indent=2)}")

Model saved to: /content/drive/MyDrive/J26-DS-320/datasets/fall_cnn_lstm_v1.pt
Results saved to: /content/drive/MyDrive/J26-DS-320/datasets/cnn_lstm_results.json

Summary: {
  "model": "CNN-LSTM (3 conv layers + 2 LSTM layers)",
  "training": {
    "epochs_trained": 15,
    "best_val_f1": 0.75,
    "train_sequences": 49,
    "val_sequences": 10,
    "test_sequences": 11
  },
  "test_metrics": {
    "precision": 1.0,
    "recall": 0.8,
    "f1": 0.8888888888888888,
    "false_alarm_rate": 0.0,
    "tp": 4,
    "fp": 0,
    "fn": 1,
    "tn": 6
  },
  "baseline_f1": 0.675,
  "improvement": 0.2138888888888888,
  "missed_sequences": [
    "fall-07-cam0-rgb"
  ]
}
